# M17 — Attention transforms tokens; a model card constrains claims

<!-- paper-first -->
### Research question

**Reading:** [PM05](../../curriculum/papers/modeling.md#pm05). Review the assigned figure or result before starting the lesson.

**Question:** How do tokens and attention connect to the paper's atlas-free claim and its evaluation limits?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

A transformer operates on tokens represented as vectors. In an imaging model, tokens may come from patches of a two-dimensional slice or three-dimensional volume. Patch creation changes the representation and may lose fine spatial detail depending on patch size and embedding. An LLM chat interface and a specialist image transformer are distinct systems with different inputs; calling a chat model does not automatically execute an MRI encoder or inspect a NIfTI affine.

Scaled dot-product attention constructs queries, keys, and values from the token vectors. A query compares with keys, scores are scaled, and softmax converts the scores into weights that sum to one across the selected keys. The output is a weighted combination of value vectors. Attention weights are internal mixing coefficients, not calibrated disease probabilities, causal effects, or complete explanations of a network prediction.

Without positional information, this attention operation is equivariant to a consistent permutation of tokens: reorder the tokens and the outputs reorder correspondingly. Pooling such outputs removes order information. Positional embeddings or other spatial mechanisms allow a model to distinguish locations. The lesson tests that property numerically, then shows that adding positions changes the result when content is moved between fixed locations. This is a concrete reason to audit patch order and positional conventions.

A complete transformer adds mechanisms such as residual connections, normalization, feed-forward blocks, and often multiple attention heads and layers. This core executes the attention transformation and its invariance check only. It does not train a transformer or reproduce a foundation model. The pinned Neuromatch attention tutorial is the deeper architecture and training extension; its example domain is language, so its dataset assumptions do not automatically transfer to brain imaging.

For a real imaging checkpoint, record the authors, model identifier, exact revision, code license, weight terms, pretraining population, modality, channel order, preprocessing, target task, and evaluation cohorts. Unknowns remain unknown. A publicly downloadable file does not itself establish unrestricted reuse or reliable generalization. The BrainSegFounder author card distinguishes UK Biobank-derived weight conditions from its separately licensed research code; do not collapse these into one open-source label.

Foundation describes a pretraining and reuse ambition, not a guarantee of performance on every domain. An encoder pretrained on one population, resolution, or contrast can fail under another. Before deployment, compare a simple baseline, check participant overlap with pretraining where possible, and evaluate independent sites and relevant subgroups. Ask AI to supply evidence and executed outputs for these claims rather than inferring capability from a model name or a plausible saliency image.

## Transformation contract

Patch-token matrix → Q, K, V projections → row-normalized attention weights → mixed token vectors. Token mixing preserves output dimensionality but generally loses invertible access to original values; pooling loses order without an additional positional mechanism.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.special import softmax
rng=np.random.default_rng(417)
tokens=rng.normal(size=(6,4));Wq=rng.normal(size=(4,3));Wk=rng.normal(size=(4,3));Wv=rng.normal(size=(4,5))
def attention(X):
    Q,K,V=X@Wq,X@Wk,X@Wv
    weights=softmax(Q@K.T/np.sqrt(Q.shape[1]),axis=1)
    return weights@V,weights
out,weights=attention(tokens)
assert out.shape==(6,5) and np.allclose(weights.sum(axis=1),1)
perm=np.array([2,0,5,1,4,3]);reordered,_=attention(tokens[perm])
print('Permutation equivariance error:',np.max(np.abs(reordered-out[perm])))
assert np.allclose(reordered,out[perm])
assert np.allclose(reordered.mean(axis=0),out.mean(axis=0))


Permutation equivariance error: 1.7763568394002505e-15


In [2]:
positions=rng.normal(0,.4,tokens.shape)
positioned,_=attention(tokens+positions)
moved,_=attention(tokens[perm]+positions)
print('Moving content between fixed positions changes pooled output:',np.linalg.norm(moved.mean(0)-positioned.mean(0)))
assert not np.allclose(moved.mean(0),positioned.mean(0))
card={'model':'BrainSegFounder','weights_downloaded':False,'inference_executed':False,
      'weight_terms':'UK Biobank MTA; verify access with author card',
      'code_license':'GPL-3.0 in author repository','checkpoint_revision':'not selected',
      'our_target_domain_validation':'not performed'}
print(card)
assert card['inference_executed'] is False and card['checkpoint_revision']=='not selected'


Moving content between fixed positions changes pooled output: 0.3570793276521015
{'model': 'BrainSegFounder', 'weights_downloaded': False, 'inference_executed': False, 'weight_terms': 'UK Biobank MTA; verify access with author card', 'code_license': 'GPL-3.0 in author repository', 'checkpoint_revision': 'not selected', 'our_target_domain_validation': 'not performed'}


## Deliberate failure and repair

Without positions, moving tokens only permutes token outputs and leaves their pooled representation unchanged. If the scientific target depends on location, that information can be lost. Repair the architecture and input convention, not the evaluation labels. A second failure is filling an unknown checkpoint revision or validation field with an AI guess.

## Your investigation

Draw the shapes of Q, K, the score matrix, V, and the output. Explain why softmax row sums are not diagnostic probabilities. Complete an author-card audit and identify which claims require an actual checkpoint run and independent data. Compare the claims supported by this attention calculation with those supported by a trained imaging model.

## Transfer to real neuroimaging

For MRI transformers, additionally verify 3D orientation, voxel spacing, patch layout, and modality-specific normalization. No checkpoint is selected, downloaded, fine-tuned, or evaluated in this notebook. The provenance audit must precede any separate licensed-model experiment.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA DL: attention and transformers](https://github.com/NeuromatchAcademy/course-content-dl/blob/caba36c513fb8139ac3c9e7503f7a769dadde25e/tutorials/W3D1_AttentionAndTransformers/student/W3D1_Tutorial1.ipynb)
- [BrainSegFounder author model card](https://huggingface.co/smilelab/BrainSegFounder)
- [BrainSegFounder author research code](https://github.com/lab-smile/BrainSegFounder)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Does attention weight equal causal importance? **No; it is a model-internal mixing weight.**
2. Has this notebook run BrainSegFounder? **No; it audits the model-card distinction and runs only synthetic attention.**

### Return to the research question

Revisit [PM05](../../curriculum/papers/modeling.md#pm05) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
